# Using the 96 head

Some liquid handling robots have a 96 head, which can be used to pipette 96 samples at once. On the Hamilton STAR, the 96 head is an optional module. After {meth}`~pylabrobot.hamilton.liquid_handlers.star.star.STAR.setup`, it is available as `star.head96` (or `None` if not installed).

This notebook shows how to use the {class}`~pylabrobot.capabilities.liquid_handling.head96.Head96` capability to pick up tips, aspirate, dispense, and work with 384-well plate quadrants.

## Setup

Import {class}`~pylabrobot.hamilton.liquid_handlers.star.star.STAR` and create a {class}`~pylabrobot.resources.hamilton.decks.STARLetDeck`. After `setup()`, `star.head96` is a {class}`~pylabrobot.capabilities.liquid_handling.head96.Head96` instance if the hardware is installed.

In [1]:
from pylabrobot.hamilton.liquid_handlers.star import STAR
from pylabrobot.resources.hamilton import STARLetDeck

deck = STARLetDeck()
star = STAR(deck=deck)
await star.setup()

2026-04-03 16:52:56,509 - pylabrobot.io.usb - INFO - Finding USB device...
2026-04-03 16:52:56,513 - pylabrobot.io.usb - INFO - Found USB device.
2026-04-03 16:52:56,513 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0


## Creating the deck layout

Assign a tip carrier with a 96-tip rack and a plate carrier with a 96-well plate.

In [2]:
from pylabrobot.resources import (
    TIP_CAR_480_A00,
    PLT_CAR_L5AC_A00,
    hamilton_96_tiprack_50uL,
    Cor_96_wellplate_360ul_Fb,
)

tip_carrier = TIP_CAR_480_A00(name="tip_carrier")
tip_carrier[1] = tip_rack = hamilton_96_tiprack_50uL(name="tip_rack")
deck.assign_child_resource(tip_carrier, rails=3)

plt_carrier = PLT_CAR_L5AC_A00(name="plt_carrier")
plt_carrier[0] = plate = Cor_96_wellplate_360ul_Fb(name="plate")
deck.assign_child_resource(plt_carrier, rails=10)

## Liquid handling with the 96 head

Liquid handling with the 96 head is very similar to what you would do with individual channels. The methods live on `star.head96` and take {class}`~pylabrobot.resources.tip_rack.TipRack`s and {class}`~pylabrobot.resources.plate.Plate`s as arguments, as opposed to individual `TipSpot`s and `Well`s used with `star.pip`.

In [ ]:
await star.head96.pick_up_tips(tip_rack)

For aspirations and dispenses, a single volume is passed because all 96 channels move in unison.

```{note}
Only single-volume aspirations and dispenses are supported because all robots that are currently implemented only support single-volume operations. When we add support for robots that can do variable-volume, this will be updated.
```

In [ ]:
await star.head96.aspirate(plate, volume=50)

In [ ]:
await star.head96.dispense(plate, volume=50)

In [ ]:
await star.head96.return_tips()

## Quadrants

96 heads can also be used to pipette quadrants of a 384-well plate. A 384-well plate is laid out as four interleaved 96-well quadrants. Use {meth}`~pylabrobot.resources.plate.Plate.get_quadrant` to select the wells for a given quadrant (1 through 4).

![quadrants](img/96head/quadrants.png)

In [ ]:
from pylabrobot.resources import BioRad_384_wellplate_50uL_Vb

plt_carrier[1] = plate384 = BioRad_384_wellplate_50uL_Vb(name="plate384")

In [ ]:
await star.head96.pick_up_tips(tip_rack)

In [ ]:
await star.head96.aspirate(plate384.get_quadrant("tl"), volume=10)

In [ ]:
await star.head96.dispense(plate384.get_quadrant("bl"), volume=10)

In [ ]:
await star.head96.aspirate(plate384.get_quadrant("tr"), volume=10)

In [ ]:
await star.head96.dispense(plate384.get_quadrant("br"), volume=10)

In [ ]:
await star.head96.return_tips()

## Backend parameters

For STAR-specific tuning, pass `backend_params` to any 96-head operation. The available parameter classes are:

- {class}`~pylabrobot.hamilton.liquid_handlers.star.head96_backend.STARHead96Backend.PickUpTips96Params`
- {class}`~pylabrobot.hamilton.liquid_handlers.star.head96_backend.STARHead96Backend.DropTips96Params`
- {class}`~pylabrobot.hamilton.liquid_handlers.star.head96_backend.STARHead96Backend.Aspirate96Params`
- {class}`~pylabrobot.hamilton.liquid_handlers.star.head96_backend.STARHead96Backend.Dispense96Params`

For example, to use LLD:

In [ ]:
from pylabrobot.hamilton.liquid_handlers.star.head96_backend import STARHead96Backend

await star.head96.pick_up_tips(tip_rack)
await star.head96.aspirate(
  plate,
  volume=50,
  backend_params=STARHead96Backend.Aspirate96Params(
    use_lld=True,
  )
)
await star.head96.dispense(plate, volume=50)
await star.head96.return_tips()

## Stamping

Stamping lets you aspirate from one plate and dispense to another in a single call. This is convenient for plate-to-plate transfers where the source and target have the same well layout.

In [ ]:
target_plate = Cor_96_wellplate_360ul_Fb(name="target_plate")
plt_carrier[2] = target_plate

await star.head96.pick_up_tips(tip_rack)
await star.head96.stamp(plate, target_plate, volume=50)
await star.head96.return_tips()

## Discarding tips

Use `discard_tips()` to drop tips into the trash instead of returning them to the tip rack. The 96-head trash location on the deck is found automatically.

In [ ]:
await star.head96.pick_up_tips(tip_rack)
await star.head96.discard_tips()

## Direct movement

For low-level control, access the backend directly via `star.driver.head96`. This gives you absolute positioning, axis-specific moves, safety moves, position queries, and tip presence checks.

In [ ]:
from pylabrobot.resources import Coordinate

await star.driver.head96.move_to_coordinate(Coordinate(x=200, y=100, z=300))

Move individual axes:

```{warning}
Direct movement is very powerful but also very dangerous. Always make sure you know exactly where the head will move to and that there are no obstacles in the way.
```

In [ ]:
await star.driver.head96.move_y(y=200)
await star.driver.head96.move_z(z=300)

Move to the safe Z height:

In [ ]:
await star.driver.head96.move_to_z_safety()

Query the current position and tip presence:

In [ ]:
position = await star.driver.head96.request_position()
tips_present = await star.driver.head96.request_tip_presence()

Park the 96-head:

In [ ]:
await star.driver.head96.park()

## Dispensing drive control

The dispensing (plunger) drive can be controlled directly to move to absolute volume positions or query the current position in microliters.

In [ ]:
# Move plunger to an absolute volume position (in uL)
await star.driver.head96.dispensing_drive_move_to_position(position=100)

# Query current plunger position in uL and mm
current_pos = await star.driver.head96.dispensing_drive_request_position_uL()
current_pos = await star.driver.head96.dispensing_drive_request_position_mm()

## Trough support

The 96 head can aspirate from a single trough container with all 96 channels simultaneously. Just pass the trough resource to `aspirate` the same way you would pass a plate.

In [9]:
from pylabrobot.resources import Trough

trough = Trough(name="trough", size_x=127.0, size_y=86.0, size_z=44.0, max_volume=300_000, material_z_thickness=1)
plt_carrier[3] = trough

await star.head96.pick_up_tips(tip_rack)
await star.head96.aspirate(trough, volume=50)
await star.head96.dispense(plate, volume=50)
await star.head96.return_tips()

## Teardown

In [ ]:
await star.stop()